In [5]:
from typing import Annotated, Optional
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage, AIMessage, trim_messages
import operator
from langsmith import utils
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from grooagents.tools.analysis import (
    generate_embeddings,
    run_clustering,
    generate_feature_importance,
    shap_to_nlp,
    kb_query,
    visualize,
    table_lookup,
    list_cache_files,
    peek_csv
)
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv('../.env')

# Define the state type
class State(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    reflection_count: Annotated[int, operator.add]

# Initialize the model and tools
model = ChatOpenAI(model="gpt-4.1", temperature=0.7)
tools = [generate_embeddings, run_clustering, generate_feature_importance, shap_to_nlp, kb_query, visualize, table_lookup, peek_csv, list_cache_files]
tools_names = {t.name: t for t in tools}
model = model.bind_tools(tools)

def run_planner(state: State):
    """Run the planner to generate a structured plan"""
    messages = state['messages'][-5:]  # Keep last 5 messages
    
    planning_prompt = """You are a planning agent. Your task is to create a detailed plan to answer the user's query.
Available tools:
- peek_csv: Show the structure and first few rows of a CSV file
- table_lookup: Search through CSV files for specific information. Mention the most granular name. 
- kb_query: Query the knowledge base for information
- list_cache_files: List available CSV files in cache
- generate_embeddings: Generate embeddings for data analysis
- run_clustering: Run clustering on embeddings
- generate_feature_importance: Analyze feature importance
- visualize: Create visualizations
- shap_to_nlp: Convert SHAP analysis to natural language

Create a markdown plan that:
1. Lists the steps needed to answer the query
2. Specifies which tools to use and why
3. Describes what information we expect to get from each step
4. Identifies potential gaps in information

Do NOT generate tool calls. Just create a clear, structured plan."""
    
    messages.append(SystemMessage(content=planning_prompt))
    response = model.invoke(messages)
    return {'messages': [response]}

def run_executor(state: State):
    """Execute the plan by interpreting the planner's output and making tool calls or returning a summary"""
    messages = state['messages']
    execution_prompt = """
You are an execution agent responsible for fulfilling the user's request completely and accurately. Your task is to:

1. CAREFULLY READ both:
   - The original user query
   - The planner's markdown plan
   - Ensure you are not missing any parts of the user's request and the plan

2. Track what has been completed and what remains:
   - Keep a mental checklist of all aspects of the user's request
   - Ensure NO part of the request is overlooked
   - If multiple data sources are mentioned (CSV and Knowledge Base), ensure BOTH are checked

3. Execute tools in a logical order:
   - Always start with list_cache_files to understand available data
   - For CSV files:
     * Use peek_csv first to understand the structure
     * Then use table_lookup with precise column names
   - For knowledge base queries:
     * Break down complex queries into specific aspects
     * Use kb_query for each distinct aspect

4. After each tool response, REFLECT on:
   - What information has been gathered
   - What parts of the user's request remain unfulfilled
   - Whether the current information is sufficient and accurate
   - If additional tool calls are needed to:
     * Fill information gaps
     * Cross-verify data
     * Get more specific details

5. Before making tool calls, ensure they:
   - Directly contribute to answering the user's query
   - Are not redundant with previous calls
   - Use precise and specific query parameters

6. NEVER DIRECTLY ANSWER THE USER'S QUERY. THE TOOLS ARE FOR YOU TO GATHER INFORMATION.

Available tools:
- list_cache_files: List available CSV files in cache
- peek_csv: Show the structure and first few rows of a CSV file
- table_lookup: Search through CSV files for specific information
- kb_query: Query the knowledge base for information
- generate_embeddings: Generate embeddings for data analysis
- run_clustering: Run clustering on embeddings
- generate_feature_importance: Analyze feature importance
- visualize: Create visualizations
- shap_to_nlp: Convert SHAP analysis to natural language

CRITICAL CHECKLIST before proceeding:
1. Have you checked BOTH CSV files and knowledge base if the query requires it?
2. Have you gathered ALL the information requested by the user?
3. Are there any aspects of the query that haven't been addressed?
4. Is the information specific and detailed enough?
5. Have you cross-verified important information when possible?

Do NOT generate a summary - that will be handled by the summarize node.
Instead, make tool calls until you have gathered ALL necessary information to fully answer the user's query.

If making tool calls:
- Be specific in your queries
- Use proper column names for CSV searches
- Break down complex knowledge base queries into specific aspects

If you've gathered all necessary information:
- Double-check that ALL aspects of the user's query have been addressed
- Verify that both CSV and knowledge base have been consulted if required or if the user has mentioned to use both
- Ensure the information is complete and specific enough
"""
    messages.append(SystemMessage(content=execution_prompt))
    response = model.invoke(messages)
    return {'messages': [response]}

def summarize(state: State):
    """Generate a final summary based on all gathered information."""
    messages = state['messages']
    summary_prompt = """
You are a summarization agent. Your task is to generate a comprehensive, well-structured markdown summary based on all the information and tool results collected so far.

Your summary should include:
- A clear title
- Introduction section
- Main findings sections with subheadings
- Data points and statistics in bullet points or tables
- Conclusion section
- Sources/references section

Do NOT make any tool calls. Only return the summary.
"""
    messages.append(SystemMessage(content=summary_prompt))
    response = model.invoke(messages)
    return {'messages': [response]}

def execute_tools(state: dict) -> dict:
    """Execute any tools called by the executor"""
    tool_calls = state['messages'][-1].tool_calls
    tool_messages = []

    for t in tool_calls:
        if t['name'] not in tools_names:
            result = {"messages": "Error: No such tool"}
        else:
            tool_input = t['args'].copy()
            tool_input['state'] = state
            result = tools_names[t['name']].invoke(tool_input)
            # Merge extra keys into state
            additional_updates = {}
            for key, value in result.items():
                if key != 'messages':
                    additional_updates[key] = value
            state.update(additional_updates)

        tm = ToolMessage(
            tool_call_id=t['id'],
            name=t['name'],
            content=result.get('messages', '')
        )
        tool_messages.append(tm)

    # Add tool messages to state
    state['messages'].extend(tool_messages)
    # Increment reflection count when tools are executed
    state['reflection_count'] = state.get('reflection_count', 0) + 1
    return state

def should_continue(state: State) -> bool:
    """Check if we should continue execution"""
    # If the last message is a tool message and we haven't exceeded reflection limit
    return isinstance(state['messages'][-1], ToolMessage) and state.get('reflection_count', 0) < 5

def tool_exists(state: State) -> bool:
    """Check if the last message contains tool calls"""
    return len(state['messages'][-1].tool_calls) > 0

# Build the graph
graph_builder = StateGraph(State)

# Add nodes
graph_builder.add_node("planner", run_planner)
graph_builder.add_node("executor", run_executor)
graph_builder.add_node("tools", execute_tools)
graph_builder.add_node("summarize", summarize)

# Add edges
graph_builder.add_edge("planner", "executor")
graph_builder.add_conditional_edges(
    "executor",
    tool_exists,
    {True: "tools", False: "summarize"}
)
graph_builder.add_conditional_edges(
    "tools",
    should_continue,
    {True: "executor", False: "summarize"}
)
graph_builder.add_edge("summarize", END)

graph_builder.set_entry_point("planner")

# Compile the graph
workflow = graph_builder.compile()

SYSTEM_PROMPT = """You are an agentic AI assistant with access to various tools for data analysis and information retrieval. Your task is to help users by:

1. Understanding their query and breaking it down into manageable steps
2. Using the appropriate tools to gather information
3. Combining information from multiple sources to provide comprehensive answers
4. Providing clear, well-structured responses

Available tools:
- peek_csv: Show the structure and first few rows of a CSV file
- table_lookup: Search through CSV files for specific information
- kb_query: Query the knowledge base for information
- list_cache_files: List available CSV files in cache
- generate_embeddings: Generate embeddings for data analysis
- run_clustering: Run clustering on embeddings
- generate_feature_importance: Analyze feature importance
- visualize: Create visualizations
- shap_to_nlp: Convert SHAP analysis to natural language

When answering queries:
1. First check what CSV files are available using list_cache_files
2. Use peek_csv to understand the structure of the CSV files and then use table_lookup to search through relevant files
3. Use kb_query to get additional context from the knowledge base (Always use this if the user hasn't mentioned to use just CSV or Knowledge Base)
4. Combine information from all sources to provide a comprehensive answer

Your responses should be:
1. Clear and well-structured
2. Based on actual data from the tools
3. Include relevant context and explanations
4. Cite sources when possible

Remember to:
1. Always check available files first
2. Use multiple sources when appropriate
3. Provide context for your answers
4. Be transparent about your information sources"""

def run_agent(query: str, max_reflections: int = 5):
    """Run the agent with a given query"""
    # Initialize state with system message and user query
    initial_state = {
        "messages": [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=query)
        ],
        "reflection_count": 0
    }
    
    # Run the workflow
    final_response = None
    for output in workflow.stream(initial_state):
        print(output)
        if "summarize" in output:
            last_message = output["summarize"]["messages"][-1]
            final_response = last_message.content
    
    return final_response if final_response else "No response generated"

In [ ]:
query = "Hello"

In [ ]:
response = run_agent(query)

In [ ]:
print(response)